# PREreview staged data pipeline

This notebook executes the same functions as the command-line scripts. Each stage writes a durable intermediate artifact, so you can inspect, stop, resume, or replace one metadata source without repeating the entire crawl.

## Pipeline structure

1. **Review collection:** PREreview/Zenodo records only.
2. **Metadata enrichment:** resolve the explicitly linked DOI/arXiv targets.
3. **Dataset assembly:** group versions, assign rounds, deduplicate reviews, and write CSV.
4. **Validation:** check the final eight-column schema and semantic constraints.

In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pipeline_stages import (
    PipelineConfig, default_paths, stage1_collect_reviews,
    stage2_resolve_metadata, stage3_build_dataset,
    stage4_validate_dataset,
)

## Parameters

Use a small `LIMIT` while inspecting the workflow. Set it to `300` for the complete-thread target run. `MAX_PAGES=100` currently covers up to 2500 Zenodo records at 25 records per page. For a deliberately partial live test, set `allow_partial_scan=True`; full scans remain the default.

In [ ]:
ROOT = REPO_ROOT / 'data' / 'pipeline'
STATE_DIR = ROOT / 'state'
LIMIT = 50
MAX_PAGES = 100
CHECKPOINT_EVERY = 25

paths = default_paths(ROOT)
config = PipelineConfig.from_env(
    state_dir=str(STATE_DIR),
    checkpoint_every=CHECKPOINT_EVERY,
    field_policy='metadata',
    sampling_policy='coverage',
    use_datacite=True,
    use_crossref=True,
    use_openalex=False,
)
paths

## Stage 1 — collect review data only

The reviewed identifier is retained because it is part of the explicit Zenodo relation. No external paper metadata is queried in this stage.

In [ ]:
stage1_stats = stage1_collect_reviews(
    output=paths['reviews'],
    stats_output=paths['reviews_stats'],
    max_pages=MAX_PAGES,
    config=config,
)
stage1_stats

In [ ]:
stage1 = json.loads(paths['reviews'].read_text(encoding='utf-8'))
{
    'families': len(stage1['families']),
    'first_family': stage1['families'][0] if stage1['families'] else None,
}

## Stage 2 — resolve DOI/arXiv metadata

DataCite and Crossref are enabled by default. OpenAlex remains disabled unless explicitly enabled. Set `CROSSREF_MAILTO` in the environment before running a large job.

In [ ]:
stage2_stats = stage2_resolve_metadata(
    reviews_input=paths['reviews'],
    output=paths['metadata'],
    stats_output=paths['metadata_stats'],
    config=config,
    retry_missing=False,
)
stage2_stats

In [ ]:
stage2 = json.loads(paths['metadata'].read_text(encoding='utf-8'))
status_counts = {}
for item in stage2['records'].values():
    status_counts[item['status']] = status_counts.get(item['status'], 0) + 1
status_counts

## Stage 3 — assemble strict and extended datasets

This step consumes only the two frozen artifacts above. It does not make network requests. The extended CSV preserves the target DOI for every review round.

In [ ]:
stage3_stats = stage3_build_dataset(
    reviews_input=paths['reviews'],
    metadata_input=paths['metadata'],
    output=paths['csv'],
    extended_output=paths['extended_csv'],
    audit_output=paths['audit'],
    dedup_output=paths['dedup'],
    stats_output=paths['build_stats'],
    limit=LIMIT,
    config=config,
)
stage3_stats

In [ ]:
import csv

with paths['csv'].open(encoding='utf-8-sig', newline='') as file:
    rows = list(csv.DictReader(file))
rows[:2]

## Stage 4 — validate the final CSV

In [ ]:
validation = stage4_validate_dataset(
    csv_input=paths['csv'],
    audit_input=paths['audit'],
    report_output=paths['validation'],
    expected=LIMIT,
)
validation

## Resume and refresh behavior

- Rerun a cell with the same paths to resume from caches/checkpoints.
- Set `refresh_zenodo=True` in `PipelineConfig` to fetch a fresh review snapshot.
- Set `refresh_metadata=True` to re-query metadata providers.
- Use a different `STATE_DIR` for a fully independent run.